# Stage 1 — Data Annotation

**Goal:** Take raw, unlabelled text samples and produce a clean, labelled dataset ready for LLM evaluation and fine-tuning.

We will:
1. Load raw text samples
2. Define annotation guidelines and label schema
3. Annotate samples manually (simulated) and with a rule-based baseline
4. Compute inter-annotator agreement (Cohen's Kappa)
5. Resolve conflicts and export a final `annotated_dataset.jsonl` file

**Task:** Sentiment classification — label each sentence as `positive`, `negative`, or `neutral`.

## 1. Install & import dependencies

In [1]:
# Run once to install required packages
# !pip install scikit-learn pandas nltk

In [2]:
import json
import random
import pandas as pd
from sklearn.metrics import cohen_kappa_score
#cohen_kappa_score is a statistical metric used to measure inter-rater reliability (or inter-annotator agreement) for qualitative or categorical items. 
#It calculates the degree of agreement between two raters who classify items into mutually exclusive categories, 
#while specifically accounting for the likelihood of agreement occurring by chance

random.seed(42)
print('Dependencies loaded.')

Dependencies loaded.


## 2. Raw data — unlabelled samples

In a real project you would load a CSV / database export here.
We create 30 representative sentences inline.

In [3]:
RAW_TEXTS = [
    # positive
    'The product exceeded all my expectations — absolutely love it!',
    'Fantastic customer service, will definitely buy again.',
    'Best purchase I have made this year.',
    'The team was incredibly helpful and responsive.',
    'Shipping was fast and the packaging was perfect.',
    'I am thrilled with the quality of this item.',
    'Really impressed — five stars without hesitation.',
    'Works exactly as described, very happy.',
    'Great value for money, highly recommend.',
    'Outstanding experience from start to finish.',
    # negative
    'Terrible quality — broke after two days.',
    'The worst customer support I have ever experienced.',
    'Product arrived damaged and nobody responded to my complaint.',
    'Total waste of money, deeply disappointed.',
    'Instructions were incomprehensible and the item does not work.',
    'Very poor build quality, feels extremely cheap.',
    'Late delivery with zero communication from the seller.',
    'Returned immediately — unusable right out of the box.',
    'Absolutely not worth the price, avoid this product.',
    'Misleading description, nothing like what was advertised.',
    # neutral / mixed
    'The item arrived on time.',
    'It does what it says on the box.',
    'Average product, nothing special.',
    'Packaging was fine but the product itself is mediocre.',
    'Some features are good, others are lacking.',
    'Delivery took longer than expected but the product is okay.',
    'Not bad, not great — sits somewhere in the middle.',
    'Works as expected, no complaints but no excitement either.',
    'The colour is different from the photo but functionally fine.',
    'Acceptable for the price point.',
]

print(f'Loaded {len(RAW_TEXTS)} raw samples.')
pd.DataFrame({'text': RAW_TEXTS}).head(5)

Loaded 30 raw samples.


,text
0,The product exceeded all my expectations — abs...
1,"Fantastic customer service, will definitely bu..."
2,Best purchase I have made this year.
3,The team was incredibly helpful and responsive.
4,Shipping was fast and the packaging was perfect.


## 3. Annotation guidelines

| Label | Definition | Examples |
|---|---|---|
| `positive` | Overall sentiment is favourable; satisfaction, praise, delight | *'Best purchase ever'*, *'Highly recommend'* |
| `negative` | Overall sentiment is unfavourable; disappointment, anger, frustration | *'Total waste of money'*, *'Broke after two days'* |
| `neutral` | No clear positive or negative lean; factual, mixed, or indifferent | *'Arrived on time'*, *'Does what it says'* |

**Rules:**
- Label the *overall* sentiment, not a single word.
- Mixed sentences that lean positive → `positive`; lean negative → `negative`; truly balanced → `neutral`.
- Sarcasm should be labelled as the *intended* sentiment (usually negative).

## 4. Annotator A — rule-based baseline

In [4]:
#Simple annotator. We search the sentences if positive words are more than negative then def returns positive
#otherwise negative
#if positive words are even then returns neutral

POSITIVE_KEYWORDS = [
    'love', 'fantastic', 'best', 'great', 'excellent', 'outstanding',
    'thrilled', 'impressed', 'perfect', 'happy', 'highly recommend',
    'five stars', 'exceeded', 'amazing', 'wonderful',
]

NEGATIVE_KEYWORDS = [
    'terrible', 'worst', 'damaged', 'waste', 'poor', 'avoid',
    'broken', 'incomprehensible', 'misleading', 'disappointed',
    'unusable', 'cheap', 'late', 'returned',
]

def rule_based_label(text: str) -> str:
    text_lower = text.lower()
    pos_hits = sum(1 for kw in POSITIVE_KEYWORDS if kw in text_lower)
    neg_hits = sum(1 for kw in NEGATIVE_KEYWORDS if kw in text_lower)
    if pos_hits > neg_hits:
        return 'positive'
    elif neg_hits > pos_hits:
        return 'negative'
    return 'neutral'

labels_a = [rule_based_label(t) for t in RAW_TEXTS]
print('Annotator A (rule-based) distribution:')
pd.Series(labels_a).value_counts()

Annotator A (rule-based) distribution:


positive    10
neutral     10
negative    10
Name: count, dtype: int64

## 5. Annotator B — human labels (simulated)

In a real workshop this is where each participant labels the data manually.
Here we simulate a second annotator who introduces a small, realistic noise rate.

In [7]:
#human annotator simulation
GOLD_LABELS = (
    ['positive'] * 10 +
    ['negative'] * 10 +
    ['neutral']  * 10
)

def simulate_human_annotator(gold: list, noise_rate: float = 0.10) -> list:
    """
    Simulate a human annotator who agrees with gold most of the time
    but randomly flips ~noise_rate of labels to model disagreement.
    """
    classes = ['positive', 'negative', 'neutral']
    result = []
    for label in gold:
        if random.random() < noise_rate:
            alternatives = [c for c in classes if c != label]
            result.append(random.choice(alternatives))
        else:
            result.append(label)
    return result

labels_b = simulate_human_annotator(GOLD_LABELS, noise_rate=0.10)
print('Annotator B (human-simulated) distribution:')
pd.Series(labels_b).value_counts()

Annotator B (human-simulated) distribution:


negative    11
positive    10
neutral      9
Name: count, dtype: int64

## 6. Inter-annotator agreement — Cohen's Kappa

Cohen's Kappa κ corrects for chance agreement.

| κ range | Interpretation |
|---|---|
| < 0.20 | Slight |
| 0.21 – 0.40 | Fair |
| 0.41 – 0.60 | Moderate |
| 0.61 – 0.80 | Substantial |
| 0.81 – 1.00 | Almost perfect |

In [8]:
kappa = cohen_kappa_score(labels_a, labels_b)
print(f'Cohen\'s Kappa between Annotator A and B: {kappa:.3f}')

# Show disagreements
df = pd.DataFrame({
    'text': RAW_TEXTS,
    'annotator_a': labels_a,
    'annotator_b': labels_b,
})
disagreements = df[df['annotator_a'] != df['annotator_b']].reset_index(drop=True)
print(f'\nDisagreements: {len(disagreements)} / {len(df)}')
disagreements

Cohen's Kappa between Annotator A and B: 0.800

Disagreements: 4 / 30


,text,annotator_a,annotator_b
0,The team was incredibly helpful and responsive.,neutral,positive
1,Outstanding experience from start to finish.,positive,negative
2,"Average product, nothing special.",neutral,positive
3,"Not bad, not great — sits somewhere in the mid...",positive,neutral


## 7. Conflict resolution — majority vote / gold fallback

Strategy: where both annotators agree, use their shared label.
Where they disagree, fall back to the ground-truth gold label.

In [9]:
def resolve_label(a: str, b: str, gold: str) -> tuple:
    if a == b:
        return a, 'agreement'
    return gold, 'gold_fallback'

resolved = [
    resolve_label(a, b, g)
    for a, b, g in zip(labels_a, labels_b, GOLD_LABELS)
]

final_labels  = [r[0] for r in resolved]
resolution_method = [r[1] for r in resolved]

df['final_label']       = final_labels
df['resolution_method'] = resolution_method

print('Resolution method breakdown:')
print(df['resolution_method'].value_counts().to_string())
print('\nFinal label distribution:')
print(df['final_label'].value_counts().to_string())

Resolution method breakdown:
resolution_method
agreement        26
gold_fallback     4

Final label distribution:
final_label
positive    10
negative    10
neutral     10


## 8. Dataset quality checks

In [10]:
# Check for duplicates
n_duplicates = df['text'].duplicated().sum()
print(f'Duplicate texts: {n_duplicates}')

# Check for empty strings
n_empty = (df['text'].str.strip() == '').sum()
print(f'Empty texts:     {n_empty}')

# Check class balance
print('\nClass balance:')
balance = df['final_label'].value_counts(normalize=True).mul(100).round(1)
for label, pct in balance.items():
    bar = '#' * int(pct // 2)
    print(f'  {label:10s}  {bar:25s}  {pct}%')

Duplicate texts: 0
Empty texts:     0

Class balance:
  positive    ################           33.3%
  negative    ################           33.3%
  neutral     ################           33.3%


## 9. Train / validation / test split

In [ ]:
from sklearn.model_selection import train_test_split

indices = list(range(len(df)))
train_idx, temp_idx = train_test_split(indices, test_size=0.3, random_state=42,
                                        stratify=final_labels)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.5, random_state=42,
                                        stratify=[final_labels[i] for i in temp_idx])

split_map = {i: 'train' for i in train_idx}
split_map.update({i: 'validation' for i in val_idx})
split_map.update({i: 'test' for i in test_idx})

df['split'] = df.index.map(split_map)

print('Split sizes:')
print(df['split'].value_counts().to_string())

## 10. Export annotated dataset

In [ ]:
OUTPUT_PATH = 'annotated_dataset.jsonl'

records = []
for _, row in df.iterrows():
    records.append({
        'id':     int(_),
        'text':   row['text'],
        'label':  row['final_label'],
        'split':  row['split'],
        'meta': {
            'annotator_a':        row['annotator_a'],
            'annotator_b':        row['annotator_b'],
            'resolution_method':  row['resolution_method'],
        }
    })

with open(OUTPUT_PATH, 'w') as f:
    for rec in records:
        f.write(json.dumps(rec) + '\n')

print(f'Saved {len(records)} records → {OUTPUT_PATH}')

# Preview first 3 lines
with open(OUTPUT_PATH) as f:
    for i, line in enumerate(f):
        print(json.dumps(json.loads(line), indent=2))
        if i >= 2:
            break

## ✅ Stage 1 complete

Outputs produced:
- `annotated_dataset.jsonl` — 30 labelled samples with train/val/test splits

Hand this file to **Stage 2 (notebook_02_prompting.ipynb)** for baseline evaluation with a local LLM.